In [0]:
# Import core runnable classes for building data pipelines and workflows
from langchain_core.runnables import RunnableLambda, RunnableSequence, RunnablePassthrough, RunnableParallel, RunnablePick

# Import PromptTemplate for prompt engineering with language models
from langchain.prompts import PromptTemplate

# Import Databricks-specific chat LLM integration
from databricks_langchain import ChatDatabricks

# Import hashlib for cryptographic hashing functions
import hashlib

# Import re for regular expression operations
import re

In [0]:
def encrypt_password(password: str) -> str:
    """Encrypt the given password using SHA-256."""
    return hashlib.sha256(password.encode("UTF-8")).hexdigest()

Pass your parameters through `invoke()` to call a runnable type object.
- The `invoke()` method executes the underlying function or sequence with the provided input.
- `RunnableLambda` is like a wrapper on top of a function that is called through `invoke()`
- Use `invoke()` to trigger the computation and get the result.

In [0]:
encrypt_password = RunnableLambda(encrypt_password)
print(encrypt_password.invoke("hello123"))
print(encrypt_password.batch(["hello123", "world123"]))

### RunnableLambda
- A RunnableLambda class is a wrapper applied on top of a function. It is used to apply transformations to inputs

In [0]:
def download_csv_files():
    def list_all_csv_files(url: str):
        return ["1.csv", "2.csv", "3.csv"]

    def load_all_csv_files(file_list):
        for file in file_list:
            print("Data load completed for file:", file)

    return RunnableLambda(list_all_csv_files) | RunnableLambda(load_all_csv_files)


# A RunnableLambda class is a wrapper applied on top of a function. It is used to apply transformations to inputs
# A runnable object can be called inside another function further invoked after initializing it to a variable
# In the above function, all runnables objects are executed in sequence. The output of first runnable is passed as input to the second runnable.

files = download_csv_files()
print(files.invoke("http://example.com/data"))

### RunnableSequence
- RunnableSequence is used to run runnable instances in a sequential order so that output from one runnable is carried to the next runnable instance

In [0]:
def load_all_csv_files():
    """
    In the first
    RunnableLambda(lambda x: {"file_list": list_all_csv_files(x["url"]), "n": x["n"]})
    we're:
        1. Calling list_all_csv_files(x["url"], "n": x["n"]) to get the file list using only the url from the input dict. "n": x["n"] in the input dict is never used in this function. But it is carried forward for next function call. We returned a dictionary that is further used down the lane.
        2. That "n": x["n"] is used to call chunkify like chunkify(d["file_list"], d["n"])
           This output dict (with file_list and n) is then passed to the second RunnableLambda(lambda d: chunkify(d["file_list"], d["n"])), which uses both values to create the chunks.
    """

    def list_all_csv_files(url: str):
        return ["1.csv", "2.csv", "3.csv"]

    def chunkify(file_list, n):
        result = []
        for file in file_list:
            for i in range(n):
                result.append(file.replace(".csv", f"_part{i}.csv"))
        return result

    def load_all_csv_files(file_list):
        for file in file_list:
            print(f"Data load completed for file:{file}")

    return RunnableSequence(
        RunnableLambda(
            lambda x: {"file_list": list_all_csv_files(x["url"]), "n": x["n"]}
        ),
        RunnableLambda(lambda d: chunkify(d["file_list"], d["n"])),
        RunnableLambda(load_all_csv_files)
    )


# A runnable object can be called inside another function further invoked after initializing it to a variable
# In the above function, all runnables objects are executed in sequence. The output of first runnable is passed as input to the second runnable.

files = load_all_csv_files()
print(files.invoke({"url": "http://example.com/data", "n": 2}))

### Build a pipeline using RunnableSequence 
### & RunnableLambda

In [0]:
def generate_odd_number(n: int) -> list[int]:
    """Generate a list of odd numbers up to n."""
    return [i for i in range(n) if i % 2 != 0]


def sum_of_odd_numbers(odd_numbers: list[int]) -> int:
    """Calculate the sum of a list of odd numbers."""
    return sum(odd_numbers)


def check_palindrome(s: int) -> bool:
    s = str(s)
    return s == s[::-1]


output = RunnableSequence(
    first=RunnableLambda(generate_odd_number),
    middle=[RunnableLambda(sum_of_odd_numbers)],
    last=RunnableLambda(check_palindrome),
)

# output = RunnableSequence(
#     first = RunnableLambda(generate_odd_number),
#     last = RunnableLambda(sum_of_odd_numbers)
# )

print(output.invoke(22))  # 121  is a palindrome
print(output.invoke(10))  # 25 is not a palindrome

In [0]:
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=2500)

prompt = PromptTemplate.from_template(
    template="Give me the steps create rest API connection in databricks"
)

formatted_prompt = prompt.format()

# result = llm.invoke(formatted_prompt)

# To pass a prompt in a runnable instance, use RunnableLambda and provide the prompt as input.
# If the prompt does not require any parameters, pass None to the invoke() method.

chain = RunnableLambda(lambda x: formatted_prompt) | llm
print(chain.invoke(None).content)

In [0]:
# Create a prompt template with parameters {n} and {year}
prompt = PromptTemplate.from_template(template = "Give me top {n} events on year {year}")

# Pass a dictionary of parameters to the prompt using RunnableLambda.
# The lambda unpacks the input dict and formats the prompt with the provided values.
chain = RunnableLambda(lambda x: prompt.format(**x)) | llm

print(chain.invoke({"n": 5, "year": 2025}).content)

### RunnablePassthrough

In [0]:
from datetime import datetime


def get_today(day):
    return datetime.today().strftime("%y")


prompt = PromptTemplate.from_template(template="Give me top {n} events on {day}")

# RunnablePassthrough() is used to pass the input value directly to the next step without any transformation.
# Here, it passes the input value as 'n' to the prompt template.
chain = {"day": RunnableLambda(get_today), "n": RunnablePassthrough()} | prompt | llm
print(chain.invoke(2).content)

### Build a pipeline using RunnableLambda 
### & RunnablePassthrough

In [0]:
def get_year(date: str) -> int:
    """Extract the year from a date string."""
    return int(re.search(r"\d{4}", date).group())


prompt = PromptTemplate.from_template(template="Give me top {n} events on year {year}")

# RunnablePassthrough.assign can be used to override or add parameters in the input dictionary.
# It allows you to specify new key-value pairs, where the value can be a function or a Runnable.
# The assigned values will override any existing keys in the input dict or add new ones.
# In this example:
#   - "n" is overridden by extracting it from the input dict (x["n"])
#   - "year" is set by extracting the year from the "date" field using get_year
# The resulting dict is then passed to the prompt template and LLM.

chain = (
    RunnablePassthrough.assign(
        n=lambda x: x["n"], year=RunnableLambda(lambda x: get_year(x["date"]))
    )
    | prompt
    | llm
)

print(chain.invoke({"n": 5, "date": "2025-01-01"}).content)

### RunnableParallel
- RunnableParallel are used to multiple chains in parallel

In [0]:
from pyspark.sql import functions as F


def create_dataframe(n):
    return spark.range(n)


def apply_transformation(df, n):
    return df.withColumn("id", F.col("id") + F.lit(n))


def load_dataframe(df, view):
    df.createOrReplaceTempView(view)


def view_data(view):
    print(f"Data for {view}")
    return spark.read.table(view)


chain1 = (
    RunnableLambda(
        lambda x: {
            "df": create_dataframe(x["records1"]),
            "n": x["n1"],
            "view": x["view1"],
        }
    )
    | RunnableLambda(
        lambda y: {"df": apply_transformation(y["df"], y["n"]), "view": y["view"]}
    )
    | RunnableLambda(
        lambda z: {"df": load_dataframe(z["df"], z["view"]), "view": z["view"]}
    )
    | RunnableLambda(lambda a: view_data(a["view"]))
)

chain2 = (
    RunnableLambda(
        lambda x: {
            "df": create_dataframe(x["records2"]),
            "n": x["n2"],
            "view": x["view2"],
        }
    )
    | RunnableLambda(
        lambda y: {"df": apply_transformation(y["df"], y["n"]), "view": y["view"]}
    )
    | RunnableLambda(
        lambda z: {"df": load_dataframe(z["df"], z["view"]), "view": z["view"]}
    )
    | RunnableLambda(lambda a: view_data(a["view"]))
)

# display(chain.invoke({"records1": 10, "n1": 5,"view1": "table1"}))
# display(chain.invoke({"records2": 20, "n2": 7,"view2": "table2"}))

# RunnableParallel allows you to execute multiple runnable chains in parallel.
# In this code, chain1 and chain2 are executed concurrently, each processing its own set of parameters.
parallel_chain = RunnableParallel({"result1": chain1, "result2": chain2})

output = parallel_chain.invoke(
    {
        "records1": 10,
        "n1": 5,
        "view1": "table1",
        "records2": 20,
        "n2": 7,
        "view2": "table2",
    }
)

display(output["result1"])
display(output["result2"])

In [0]:
from pyspark.sql import functions as F


def create_dataframe(n):
    return spark.range(n)


def apply_transformation(df, n):
    return df.withColumn("id", F.col("id") + F.lit(n))


def load_dataframe(df, view):
    df.createOrReplaceTempView(view)


def view_data(view):
    print(f"Data for {view}")
    return spark.read.table(view)


chain = (
    RunnableLambda(
        lambda x: {"df": create_dataframe(x["records"]), "n": x["n"], "view": x["view"]}
    )
    | RunnableLambda(
        lambda y: {"df": apply_transformation(y["df"], y["n"]), "view": y["view"]}
    )
    | RunnableLambda(
        lambda z: {"df": load_dataframe(z["df"], z["view"]), "view": z["view"]}
    )
    | RunnableLambda(lambda a: view_data(a["view"]))
)

# RunnableParallel allows you to execute multiple runnable chains in parallel.
# The .batch() method enables you to process a list of input dictionaries concurrently.
# Each input dict in the list is processed independently and in parallel by the specified chain.
# The output is a list of results, one for each input, preserving the order of the input list.
parallel_chain = RunnableParallel({"chain": chain})

output = parallel_chain.batch(
    [
        {"records": 15, "n": 7, "view": "table3"},
        {"records": 25, "n": 9, "view": "table4"},
    ]
)

display(output[0]['chain'])
display(output[1]['chain'])

### RunnablePick
- RunnablePick class represents a Runnable that selectively picks keys from a dictionary input.
- It allows you to specify one or more keys to extract from the input dictionary.
- It returns a new dictionary containing only the selected keys.

In [0]:
input_data = {
    "n1": 1,
    "n2": 2,
    "n3": 3,
    "n4": 4,
    "n5": 5
}

runnable = RunnablePick(keys = ["n1", "n3", "n5"])

print(runnable.invoke(input_data))

In [0]:
from langchain_core.runnables import RunnableBranch
print(RunnableBranch.__doc__)

### RunnableBranch
- Supply `n` number of conditions into it as arguments. If any condition is satisfied then it's associated runnable is executed. You can also execute normal functions.
- Works like if-elif-else condition in python

In [0]:
"""
Build a complex AI system that is composed of 2 chains. The first chain will address SQL queries of users. The second chain will address any item properties. Your system should decide which chain to use based on the user's input.
"""
from pyspark.sql import DataFrame

def retrieve_records(n, table):
    return spark.read.table(table).limit(n)

llm = ChatDatabricks(endpoint = "databricks-meta-llama-3-3-70b-instruct", max_tokens = 1000)

prompt_string1 = "Give me first {number} numbers in table {table}"
prompt_string2 = "Give the details of {item} in shopping site"

prompt = PromptTemplate.from_template(template = """Decide what type of prompt is it.
PROMPT: -
{prompt}
1. If this is a SQL prompt then in one word mention it as "SQL"
2. If this is an item property then in one word mention it as "ITEM"
3. If this is neither then in one word mention it as "ITEM"
""")

# chain1: Handles SQL queries by extracting the number and table from the input, retrieving records from the specified table, and returning a Spark DataFrame.
chain1 = {"item": RunnablePassthrough()} | RunnableLambda(lambda x: retrieve_records(x["item"]["number"], x["item"]["table"]))

# chain2: Handles item property queries by extracting the item from the input, formatting the prompt, and passing it to the LLM for a response.
chain2 = {"item": RunnablePassthrough()} | RunnableLambda(lambda x: x["item"]) | PromptTemplate.from_template(template = prompt_string2) | llm

# This chain (RunnableLambda(lambda x : prompt.format(**x)) | llm) decides which chain to use based on the user's input. Extract  the chain type and associate it with other inputs. Then execute RunnableBranch to execute that specific chain that satisfies the user's input.
chain = (RunnableLambda(lambda x : prompt.format(**x)) | llm) | RunnableLambda(lambda x: {"type": x.content, "number": 5, "table": "table1", "item": "cricket bat"}) | RunnableBranch(
    (lambda x: x["type"] == "SQL", chain1),
    (lambda x: x["type"] == "ITEM", chain2),
    lambda x: "Sorry, I don't have any details about it"
)
print(chain.invoke({"prompt": prompt_string1}).show() if isinstance(chain.invoke({"prompt": prompt_string1}), DataFrame) else "Sorry, I don't have any details about it")
print(chain.invoke({"prompt": prompt_string2}).content)

In [0]:
print(1)